In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
# Argumentos de estabilidad para Docker
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

try:
    # El Manager instala el driver compatible con la versión de Chrome instalada
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    
    driver.get("https://www.google.com")
    print(f" ¡CONECTADO! Título de la página: {driver.title}")
    driver.quit()

except Exception as e:
    print(f" Error: {e}")

 ¡CONECTADO! Título de la página: Google


In [6]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==============================================================================
# 1. MOTOR DE NAVEGACIÓN (Configuración de Laboratorio)
# ==============================================================================
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

# ==============================================================================
# 2. CONFIGURACIÓN DEL GRUPO (Cada grupo modifica aquí)
# ==============================================================================
# Pegar la URL del sitio asignado
URL_OBJETIVO = "https://www.investing.com/currencies/eur-usd"

# Definir las etiquetas técnicas (clases CSS) encontradas con el Inspector de Chrome
SELECTOR_CONTENEDOR = "tr"
SELECTOR_DATO_A     = "td:nth-child(2)"
SELECTOR_DATO_B     = "td:nth-child(3)"

# ==============================================================================
# 3. EJECUCIÓN DEL SCRAPING
# ==============================================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print(f"Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)
    time.sleep(10) # Tiempo de espera para carga de datos dinámicos
    
    # Capturamos todos los bloques de información
    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f"Se encontraron {len(elementos)} registros potenciales.")
    
    for item in elementos:
        try:
            # Extracción genérica de datos
            columna_a = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text
            
            # Si la fila está vacía o es el encabezado, saltamos a la siguiente
            if not columna_a or not columna_b:
                continue
            
            # Limpieza segura: conservamos números y el punto decimal
            valor_limpio = "".join(c for c in columna_b if c.isdigit() or c == '.')
            
            dataset_final.append({
                "variable_nombre": columna_a,
                "variable_valor": valor_limpio,
                "status": 1.0 
            })
        except Exception as e:
            # Si algo falla, el ciclo continúa con la siguiente fila sin detener el programa
            continue

# ==============================================================================
# 4. SALIDA DE DATOS (Para análisis posterior)
# ==============================================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_extraidos_grupo.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f"Proceso exitoso. Archivo generado: {nombre_archivo}")
        print(df.head()) # Muestra de los primeros 5 datos
    else:
        print("No se capturaron datos. Revisar los selectores CSS.")
        
finally:
    driver.quit() # Siempre cerrar el navegador al terminar
    print("Navegador cerrado.")

Conectando a la fuente de datos...
Se encontraron 261 registros potenciales.
Proceso exitoso. Archivo generado: datos_extraidos_grupo.csv
  variable_nombre variable_valor  status
0          1.1563         1.1562     1.0
1          1.1367         1.0646     1.0
2          1.1604         1.1603     1.0
3         1.15919        1.18486     1.0
4          1.1642         1.1641     1.0
Navegador cerrado.
